## Appendix 1: A short tour of OpenCV
***(Suggested time: 15-20 minutes)***

Accelerator: CPU

The goal of this lab is to show you:
- Few image preprocessing functions inside OpenCV
- Computer Vision Model Inferencing using OpenCV
- Postprocessing functions provided by OpenCV

##### **Step 0:**  Check if OpenCV is installed and obtain its version. Also, download sample image and model files for experimentation.

In [0]:
import cv2
print (cv2.__version__)

In [0]:
%%bash

wget -qq https://edge-ai-doulos.s3.us-west-2.amazonaws.com/OpenCV-example.zip
unzip -qq OpenCV-example

##### **Step 1**: Commonly used preprocessing functions

##### **Step 1(a)**: Read an image and prints its width, height and number of channels.

- OpenCV generally uses the HWC (height, width, channels) format with BGR (blue, green, red)  channel order for image representation. However, when integrating with other libraries or deep learning frameworks, you might need to handle conversions to formats like CHW (channels, height, width) or NCHW (Batch/Samples n, channels, height, width).


In [0]:
import cv2
import os

BASE_DIR = os.getcwd()

IMAGE_CLOCK = os.path.join(BASE_DIR, 'clock.jpg')

def get_image_size_channels():
  img = cv2.imread(IMAGE_CLOCK)
  width = img.shape[1]
  height = img.shape[0]
  channels = img.shape[2] if img.ndim > 2 else 1
  return width, height, channels

print(get_image_size_channels())

##### **Step 1(b)**: Resize image to dimensions 224 x 224 pixels for use with MobileNet model.

- We use a function ***cv2_imshow*** from google.colab.patches to show the resized image.

In [0]:
import cv2
import matplotlib.pyplot as plt

# Load the image
image = cv2.imread(IMAGE_CLOCK)

# Define width and height for MobileNet
width = 224
height = 224

# Resize the image
resized_image = cv2.resize(image, (width, height))

# Display the resized image
plt.imshow(cv2.cvtColor(resized_image, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show()

##### **Step 1(c)**: Change input image to HSV color space and display both images using Matplotlib.

HSV stands for Hue, Saturation, and Value. It is often used in tasks like object tracking and color segmentation because it separates color information (hue) from brightness (value) and saturation.

Hue: Represents the color (0° to 360°).
Saturation: Represents the intensity or purity of the color (0% to 100%).
Value: Represents the brightness of the color (0% to 100%).

In [0]:
import cv2
import matplotlib.pyplot as plt

image = cv2.imread(IMAGE_CLOCK)

# Resize the image
image = cv2.resize(image, (640, 640))

# Convert BGR to HSV
image_hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

# Display the HSV image
plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.title('Original BGR Image')
plt.imshow(image)

plt.subplot(1, 2, 2)
plt.title('HSV Image')
plt.imshow(image_hsv)  # HSV images look different from RGB/BGR
plt.show()

##### **Step 1(d)**: Apply Canny edge detection algorithm to detect edges in an image.

- Determine the time it takes to execute the Canny algorithm in Python.

In [0]:
import cv2
import matplotlib.pyplot as plt
from time import perf_counter

# --- Code for Canny edge detection ---
image = cv2.imread(IMAGE_CLOCK)

# Resize the image
image = cv2.resize(image, (640, 640))

image_gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

# Measure the execution time of Canny
start_time = perf_counter()
outputImg = cv2.Canny(image_gray, 200, 300)
end_time = perf_counter()

print(f"Canny edge detection took: {end_time - start_time:.4f} seconds")

# Display the images outside the time measurement part
print('Input image:')
plt.imshow(cv2.cvtColor(image_gray, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show()
print('Output image (Edges):')
plt.imshow(cv2.cvtColor(outputImg, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show()

##### **Step 1(e):** Apply brightness control to an image

In [0]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def gammaCorrection(src, gamma):
    invGamma = 1 / gamma
    table = [((i / 255) ** invGamma) * 255 for i in range(256)]
    table = np.array(table, np.uint8)
    return cv2.LUT(src, table)

img = cv2.imread(IMAGE_CLOCK)
gammaImg = gammaCorrection(img, 2.2)

image = cv2.resize(img, (300, 300))
bright_image = cv2.resize(gammaImg, (300, 300))

plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show()

plt.imshow(cv2.cvtColor(bright_image, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show()

##### **Step 2**: AI model inferencing using OpenCV

- OpenCV does model inferencing, but not model training. These examples demonstrate how OpenCV is used as an alternative to ONNX and TFLite runtime for model inferencing purposes.

##### **Step 2(a)**: OpenCV inference - cv2.dnn

- Use following OpenCV APIs to inferencing a model in ONNX format

  net = cv2.dnn.readNetFromONNX('model.onnx') **# Read Model**

  blob = cv2.dnn.blobFromImage(numpyarray, scale, size, mean) **#Load image as Numpy array**

  net.setInput(blob) **#Set image as input tensor of model**

  output = net.forward()  **#Perform forward propagation / inference**



##### Get MNIST model in ONNX format from Hugging Face

In [0]:
!wget https://huggingface.co/unity/inference-engine-mnist-12/resolve/a55699050a5faeeeaf6ddd81a5245a9b95ba3e98/mnist-12.onnx

##### **Step 2(b):** Perform inference using OpenCV on test part of MNIST dataset and determine its accuracy

In [0]:
import numpy as np
import cv2
from tensorflow.keras.datasets import mnist
import os

BASE_DIR = os.getcwd()
MNIST_MODEL = os.path.join (BASE_DIR, 'mnist-12.onnx')

# Load the frozen model in OpenCV
net = cv2.dnn.readNetFromONNX(MNIST_MODEL)

# Prepare input image
(X_train, y_train), (X_test, y_test) = mnist.load_data()

correct = 0
wrong = 0

for i in range(len(X_test)):
    img = X_test[i]
    label = y_test[i]

    # Image is scaled to 1.0 and size is set as 28 x 28 pixels
    blob = cv2.dnn.blobFromImage(img, 1.0, (28, 28))

    # Run inference
    net.setInput(blob)
    output = net.forward()
    prediction = np.argmax(output)
    if prediction == label:
        correct += 1
    else:
        wrong += 1

print("count of test samples:", len(X_test))
print("accuracy:", (correct/(correct+wrong)))

##### **Step 3.** Postprocessing of inference output using OpenCV



##### **Step 3(a)**:  OpenCV provides functions for drawing geometric shapes such as line, rectangle, and circle.

The rectangle function is used to draw a rectangle by specifying x and y coordinates for top-left and bottom-right corner. An object detection model would provide these x and y co-ordinates to draw boundaries around detected objects.

In [0]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Create a blue background image
img = np.full((200, 300, 3), (255, 0, 0), dtype=np.uint8) # Blue color in BGR is (255, 0, 0)

x1, y1 = 75, 25
x2, y2 = 200, 50
color = (0, 0, 255) # Red color in BGR is (0, 0, 255)
thickness = 3
cv2.rectangle(img, (x1, y1), (x2, y2), color, thickness)

plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show()